In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar tu paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

# 08 · Registro del modelo, exportación y esquema de entrada

En la etapa 6 hubo **muchas ejecuciones** (una por ensayo): eso es **tracking** («¿qué probamos?»). Ahora hay **un modelo
final** y se guarda otra cosa: una **versión del modelo** en el **registro** («¿qué modelo usamos y en qué versión?»), con
su firma, su ejemplo de entrada, sus métricas de la prueba y el alias **`champion`**, que dice cuál es el vigente.

| | Tracking | Registro |
|---|---|---|
| Qué guarda | cada ensayo: parámetros y métricas | el modelo final, versionado |
| Cuántos hay | muchos (57 en la etapa 6) | pocos |
| Para qué | comparar y reproducir | saber **cuál** es el modelo en uso y poder volver a una versión anterior |

Después se **exporta** el modelo desde `champion` a `models/model.joblib` (la app **no** consulta Databricks) y se escribe
el **esquema de entrada** (`models/input_schema.json`). Reglas: `especificaciones/08-registro.md` (decisiones del grupo del
5 nov 2026). **La prueba no se vuelve a evaluar.**

> **Seguridad:** un *pickle* **puede ejecutar código al cargarse**. Solo se cargan modelos que el grupo registró.

In [2]:
import hashlib
import json
import subprocess
from datetime import datetime

import joblib
import mlflow
import numpy as np
import pandas as pd
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.metrics import mean_absolute_error

from proyecto.data.split import GROUP_COL
from proyecto.features.engineering import FORBIDDEN_COLUMNS, MUNICIPALITY_CODES, feature_columns
from proyecto.inference.predict import InputValidationError, load_model, predict_score
from proyecto.inference.schema import build_schema, load_schema
from proyecto.models.evaluation import model_columns
from proyecto.models.train import TARGET
from proyecto.tracking.mlflow_utils import configure_mlflow, registered_model_name, set_experiment

ROOT = Path.cwd().resolve().parent
REPORTS_DIR, MODELS_DIR, PROCESSED_DIR = ROOT / "reports", ROOT / "models", ROOT / "data" / "processed"
FINAL_PATH = MODELS_DIR / "final_pipeline.joblib"      # el pipeline evaluado en la etapa 7 (no se sube a Git)
EXPORT_PATH = MODELS_DIR / "model.joblib"              # el que viaja con la app (sí se sube)
SCHEMA_PATH = MODELS_DIR / "input_schema.json"         # el esquema de entrada (sí se sube)
RECORD_PATH = REPORTS_DIR / "08_registro.json"
FORCE_LOCAL, LOCAL_DB = False, None


def sha256(path):
    """Huella digital de un archivo: cambia si cambia un solo byte."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

with warnings.catch_warnings(record=True) as caught:   # mostramos por qué se usó (o no) Databricks
    warnings.simplefilter("always")
    where = configure_mlflow(force_local=FORCE_LOCAL, **({"local_db": LOCAL_DB} if LOCAL_DB else {}))
for w in caught:
    if "Databricks" in str(w.message):
        print("Aviso:", w.message)
if not FORCE_LOCAL and not where.startswith("databricks"):
    raise RuntimeError(f"MLflow quedó en '{where}': el registro de esta etapa debe hacerse en Databricks.")
# Auditoría etapa 8, H1: este aviso de MLflow imprime la ruta del experimento (incluye el correo del usuario)
import logging
logging.getLogger("mlflow.tracking.fluent").setLevel(logging.WARNING)
experiment = set_experiment("saber11-modelo-final")
experiment_label = experiment.rsplit("/", 1)[-1]   # la ruta completa incluye la carpeta del usuario: no se imprime
NAME = registered_model_name("saber11_puntaje_global")
print("MLflow:", where)
print("Experimento:", experiment_label)
print("Nombre en el registro:", NAME)

MLflow: databricks (token)
Experimento: saber11-modelo-final
Nombre en el registro: workspace.default.saber11_puntaje_global


## 1. Cargar el modelo evaluado y los resultados de la prueba

- `models/final_pipeline.joblib`: se verifica que su huella sea la del modelo que produjo las predicciones de la prueba
  (`reports/prueba_huellas.json`). Si no existe o no coincide, el notebook se detiene.
- `reports/07_metricas_test.json` y `reports/07_decisiones_congeladas.json`.
- `train.joblib`: **solo** para la firma, el ejemplo de entrada, la ida y vuelta y el esquema. La prueba no se usa.

In [3]:
needed = [FINAL_PATH, REPORTS_DIR / "07_metricas_test.json", REPORTS_DIR / "07_decisiones_congeladas.json",
          REPORTS_DIR / "prueba_huellas.json", PROCESSED_DIR / "train.joblib"]
missing = [p.relative_to(ROOT).as_posix() for p in needed if not p.exists()]
if missing:
    raise FileNotFoundError(f"Faltan en este equipo: {missing}. Ejecuta antes los notebooks 03 y 07.")

fingerprints = json.loads((REPORTS_DIR / "prueba_huellas.json").read_text(encoding="utf-8"))
assert sha256(FINAL_PATH) == fingerprints["final_pipeline.joblib"], "final_pipeline.joblib no es el modelo evaluado"
final_model = joblib.load(FINAL_PATH)
test_metrics = json.loads((REPORTS_DIR / "07_metricas_test.json").read_text(encoding="utf-8"))
decisions = json.loads((REPORTS_DIR / "07_decisiones_congeladas.json").read_text(encoding="utf-8"))
print("modelo:", decisions["modelo_final"], "· huella verificada · decisiones congeladas el", decisions["fecha_y_hora"])

train = joblib.load(PROCESSED_DIR / "train.joblib")
COLUMNS = model_columns()          # las 21 columnas que usa el modelo
NUMERIC, CATEGORICAL = feature_columns()
assert len(COLUMNS) == 21 and COLUMNS == NUMERIC + CATEGORICAL
assert GROUP_COL not in COLUMNS and TARGET not in COLUMNS and not set(COLUMNS) & set(FORBIDDEN_COLUMNS)

example = train[COLUMNS].head(200).copy()                      # del ENTRENAMIENTO, nunca de la prueba
example[NUMERIC] = example[NUMERIC].astype("float64")          # un entero con vacíos no cabe en la firma
print(f"firma: {example.shape[1]} columnas (sin sede, sin identificadores, sin columnas prohibidas, sin {TARGET})")

modelo: hist gradient boosting · huella verificada · decisiones congeladas el 2026-11-05T19:53:16


firma: 21 columnas (sin sede, sin identificadores, sin columnas prohibidas, sin punt_global)


## 2. Registrar el modelo

Junto al modelo viajan los **parámetros**, las **métricas de la prueba** (con sus intervalos del 95 % por sedes, copiadas de
la etapa 7: no se recalculan), la **firma** y un **ejemplo de entrada** de 5 filas, y **etiquetas** de origen.

> **Nota técnica:** `serialization_format="cloudpickle"`; el formato por defecto rechaza un `Pipeline` completo.

In [4]:
ci = test_metrics["intervalos_95_sedes"]
metrics = {
    "test_mae": test_metrics["metricas_modelo"]["mae"], "test_rmse": test_metrics["metricas_modelo"]["rmse"],
    "test_r2": test_metrics["metricas_modelo"]["r2"],
    "test_mae_ic_inf": ci["MAE"]["modelo"][0], "test_mae_ic_sup": ci["MAE"]["modelo"][1],
    "test_rmse_ic_inf": ci["RMSE"]["modelo"][0], "test_rmse_ic_sup": ci["RMSE"]["modelo"][1],
    "test_r2_ic_inf": ci["R²"]["modelo"][0], "test_r2_ic_sup": ci["R²"]["modelo"][1],
    "baseline_mae": test_metrics["metricas_linea_base"]["mae"], "baseline_rmse": test_metrics["metricas_linea_base"]["rmse"],
    "baseline_r2": test_metrics["metricas_linea_base"]["r2"],
    "dif_mae_vs_baseline": ci["MAE"]["diferencia"],
    "dif_mae_ic_inf": ci["MAE"]["diferencia_ic"][0], "dif_mae_ic_sup": ci["MAE"]["diferencia_ic"][1],
}
params = {"modelo": decisions["modelo_final"], **{f"param_{k}": v for k, v in decisions["parametros"].items()},
          **{f"fijo_{k}": v for k, v in decisions["fijos"].items()}}
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip()
dirty = bool(subprocess.run(["git", "status", "--porcelain"], cwd=ROOT, capture_output=True, text=True).stdout.strip())
tags = {
    "etapa": "08_registro", "fecha_registro": datetime.now().isoformat(timespec="seconds"),
    "commit": commit, "cambios_sin_commit": str(dirty),
    "decisiones_congeladas": decisions["fecha_y_hora"],
    "prueba_abierta": test_metrics["prueba_abierta"].split(" · ")[0].replace("Prueba abierta el ", ""),
    "unidad_de_particion": GROUP_COL, "objetivo": TARGET,
}
signature = infer_signature(example, final_model.predict(example))

with mlflow.start_run(run_name="modelo-final") as run:
    mlflow.log_params(params)
    mlflow.log_metrics({k: float(v) for k, v in metrics.items()})
    mlflow.set_tags(tags)
    info = mlflow.sklearn.log_model(
        final_model, name="model", signature=signature, input_example=example.head(5),
        serialization_format="cloudpickle", registered_model_name=NAME,
    )
    run_id = run.info.run_id

client = MlflowClient()
version = int(info.registered_model_version)
print(f"modelo registrado: {NAME} · versión {version}")
print("commit de origen:", commit, "(con cambios sin commit)" if dirty else "")

2026/11/05 20:31:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully registered model 'workspace.default.saber11_puntaje_global'.


Created version '1' of model 'workspace.default.saber11_puntaje_global'.


modelo registrado: workspace.default.saber11_puntaje_global · versión 1
commit de origen: 3a75853 (con cambios sin commit)


## 3. Alias `champion`

Una **versión** es un número; el **alias** dice el uso: `champion` es el modelo vigente. Quien use el modelo pide
`models:/<nombre>@champion`; si mañana hay una versión mejor, se mueve el alias y nadie cambia su código.

In [5]:
client.set_registered_model_alias(NAME, "champion", str(version))
for key in ("etapa", "commit", "decisiones_congeladas", "prueba_abierta"):
    client.set_model_version_tag(NAME, str(version), key, tags[key])
champion = client.get_model_version_by_alias(NAME, "champion")
assert int(champion.version) == version
print(f"alias champion → versión {champion.version}")

alias champion → versión 1


## 4. Comprobación de ida y vuelta

Se carga **desde el registro, por alias**, y se comprueba que predice **exactamente** lo mismo (igualdad exacta, no
aproximada) que el objeto registrado, en 500 filas del entrenamiento.

In [6]:
from_registry = mlflow.sklearn.load_model(f"models:/{NAME}@champion")
sample = train[COLUMNS].head(500)
expected, obtained = final_model.predict(sample), from_registry.predict(sample)
round_trip = bool(np.array_equal(expected, obtained))
print(f"ida y vuelta: las 500 predicciones son idénticas: {round_trip} (diferencia máxima {np.abs(expected - obtained).max():.1e})")
assert round_trip, "El modelo del registro no coincide con el que se evaluó"

ida y vuelta: las 500 predicciones son idénticas: True (diferencia máxima 0.0e+00)


## 5. Exportar desde `champion` y escribir el esquema de entrada

- `models/model.joblib` se exporta **desde la versión `champion`** (el registro es la fuente de verdad).
- `models/input_schema.json` se calcula **solo con el entrenamiento**: las 21 columnas, su tipo, los valores válidos de las
  categóricas, el rango observado y la mediana de las numéricas, y el **rango válido** aprobado por el grupo (fuera de él,
  la predicción se niega). Los campos con vacíos en el entrenamiento admiten «sin respuesta» (`nullable`). En los 2
  códigos de municipio, un valor nuevo solo genera advertencia.

In [7]:
VALID_RANGE = {"edad": (13, 80), "fami_estratovivienda": (1, 6), "sin_estrato": (0, 1),
               "fami_personashogar": (1, 5), "fami_cuartoshogar": (1, 6)}
joblib.dump(from_registry, EXPORT_PATH)
schema = build_schema(train, NUMERIC, CATEGORICAL, task="regression", valid_range=VALID_RANGE,
                      warn_only_categories=MUNICIPALITY_CODES, path=SCHEMA_PATH)
print(f"exportado: {EXPORT_PATH.name} (desde champion, versión {version})")
print(f"esquema: {len(schema['numeric'])} numéricas + {len(schema['categorical'])} categóricas = "
      f"{len(schema['numeric']) + len(schema['categorical'])} columnas")
print("admiten vacío («sin respuesta»):", ", ".join(schema["nullable"]))
print("no admiten vacío:", ", ".join(c for c in COLUMNS if c not in schema["nullable"]))

for f in ("models/model.joblib", "models/input_schema.json"):
    ignored = subprocess.run(["git", "check-ignore", "-q", f], cwd=ROOT).returncode == 0
    assert not ignored, f"{f} está en .gitignore y no viajaría con la app"
    print(f"{f}: no está en .gitignore (se sube al repositorio)")

exportado: model.joblib (desde champion, versión 1)
esquema: 5 numéricas + 16 categóricas = 21 columnas
admiten vacío («sin respuesta»): edad, fami_estratovivienda, fami_personashogar, fami_cuartoshogar, cole_bilingue, cole_caracter, fami_educacionmadre, fami_educacionpadre, fami_tieneautomovil, fami_tienecomputador, fami_tieneinternet, fami_tienelavadora
no admiten vacío: sin_estrato, cole_area_ubicacion, cole_genero, cole_jornada, cole_naturaleza, cole_sede_principal, estu_genero, cole_cod_mcpio_ubicacion, estu_cod_reside_mcpio
models/model.joblib: no está en .gitignore (se sube al repositorio)
models/input_schema.json: no está en .gitignore (se sube al repositorio)


## 6. ¿El exportado es el evaluado? — sin volver a evaluar con la prueba

Decisión del grupo (opción A): **no se vuelve a predecir sobre la prueba.** Tres comprobaciones:

1. La huella de `final_pipeline.joblib` es la del modelo que produjo las predicciones de la prueba (verificada en la sección 1).
2. `models/model.joblib` predice **exactamente** igual que `final_pipeline.joblib` en **todas** las filas del entrenamiento.
3. El MAE de la prueba se recalcula con las predicciones y etiquetas **ya guardadas** en `prueba_resultados.joblib`.

In [8]:
exported = load_model(EXPORT_PATH)
X_all = train[COLUMNS]
same_everywhere = bool(np.array_equal(exported.predict(X_all), final_model.predict(X_all)))
print(f"(2) predicciones idénticas a final_pipeline.joblib en las {len(X_all):,} filas del entrenamiento: {same_everywhere}")
assert same_everywhere

stored = joblib.load(PROCESSED_DIR / "prueba_resultados.joblib")   # resultados guardados de la etapa 7 (no se predice)
assert sha256(PROCESSED_DIR / "prueba_resultados.joblib") == fingerprints["prueba_resultados.joblib"]
mae_stored = float(mean_absolute_error(stored["y"], stored["pred_model"]))
print(f"(3) MAE recalculado con las predicciones guardadas: {mae_stored:.3f} "
      f"(etapa 7: {test_metrics['metricas_modelo']['mae']:.3f})")
assert round(mae_stored, 3) == 33.316 and mae_stored == test_metrics["metricas_modelo"]["mae"]
print("→ el exportado es el mismo modelo que se evaluó con la prueba")

(2) predicciones idénticas a final_pipeline.joblib en las 57,286 filas del entrenamiento: True
(3) MAE recalculado con las predicciones guardadas: 33.316 (etapa 7: 33.316)
→ el exportado es el mismo modelo que se evaluó con la prueba


## 7. La predicción valida antes de responder

Tres casos con filas del **entrenamiento**: una normal, una incompleta («sin respuesta» en campos que lo admiten) y dos
imposibles (una edad fuera de 13–80 y una categoría que no existe). Las dos primeras responden; las imposibles **se niegan
y explican qué corregir**.

In [9]:
schema = load_schema(SCHEMA_PATH)
normal = train[COLUMNS].iloc[[0]].copy()
incomplete = normal.copy()
incomplete[[c for c in ("edad", "fami_educacionmadre", "fami_tieneinternet") if c in schema["nullable"]]] = np.nan
print("normal:     ", round(float(predict_score(normal, exported, schema)["puntaje_estimado"].iloc[0]), 2))
print("incompleta: ", round(float(predict_score(incomplete, exported, schema)["puntaje_estimado"].iloc[0]), 2))
for label, bad in (("edad 200", normal.assign(edad=200.0)), ("naturaleza «PRIVADA»", normal.assign(cole_naturaleza="PRIVADA"))):
    try:
        predict_score(bad, exported, schema)
        print(f"{label}: respondió (no debía)")
    except InputValidationError as error:
        print(f"{label}: se niega → {error}")

normal:      184.92
incompleta:  192.84
edad 200: se niega → edad: debe estar entre 13 y 80
naturaleza «PRIVADA»: se niega → cole_naturaleza: categoría no vista en el entrenamiento (PRIVADA); elige uno de los valores válidos


## 8. Guardar la constancia

`reports/08_registro.json` (sin correos ni direcciones del espacio de trabajo). El informe `reports/08_registro.md` se
escribe con estas cifras.

In [10]:
record = {
    "fecha_y_hora": tags["fecha_registro"], "donde": where, "experimento": experiment_label,
    "modelo_registrado": NAME, "version": version, "alias": "champion", "ejecucion": run_id,
    "commit": commit, "cambios_sin_commit": dirty,
    "ida_y_vuelta_identica_500_filas": round_trip,
    "exportado_desde": f"models:/{NAME}@champion", "exportado_igual_a_evaluado_en_train": same_everywhere,
    "filas_entrenamiento_comparadas": int(len(X_all)), "mae_prueba_recalculado_con_guardadas": mae_stored,
    "columnas_entrada": COLUMNS, "admiten_vacio": schema["nullable"], "rango_valido": schema["valid_range"],
    "decisiones_congeladas": decisions["fecha_y_hora"], "metricas_registradas": metrics,
}
RECORD_PATH.write_text(json.dumps(record, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
print(f"guardado: {RECORD_PATH.name} · {NAME} versión {version} · alias champion")

guardado: 08_registro.json · workspace.default.saber11_puntaje_global versión 1 · alias champion


## Entregable de esta etapa

- [x] Modelo registrado con versión, parámetros, métricas de la prueba con intervalos, firma de 21 columnas y etiquetas.
- [x] Alias `champion` y ida y vuelta exacta.
- [x] `models/model.joblib` exportado desde `champion` y `models/input_schema.json` (solo con el entrenamiento).
- [x] Comprobado que el exportado es el evaluado, sin volver a predecir sobre la prueba.
- [x] La predicción rechaza entradas imposibles.